# Konu 12 — Stüdyo asistanı

Bir tasarım stüdyosunda gün, birbirine benzemeyen küçük işlerle geçer. Sayfa Kahve için bugünün listesi:

1. Okuma gecesi 14 Kasım'da. Hangi güne denk geliyor?
2. Müşteri yazdı: "Afişi matbaaya göndermeden bir kontrol eder misin?"
3. Müşteri yine yazdı: "Bu arada, afişlerde hardal sarısını da artık istemiyoruz."
4. Yeni gönderinin metni hazır: "Hemen gel, kafemizde sınav haftası indirimi!" Müşterinin kurallarına uyuyor mu?

Her iş başka bir dosya ister: takvim, afişin kendisi, müşterinin kuralları ve notları. Konu 10'da bunlardan yalnızca
birinin, afiş kontrolünün adımlarını sırasıyla sen yazmıştın; her iş için ayrı bir hat yazmak zahmetli. Bugün bu
işlerin hepsine bakan bir **stüdyo asistanı** kuruyoruz: ona yalnızca hedefi söyleyeceksin, hangi dosyaya bakacağını o
seçecek. Adım 1–4 bu listeyi sırayla bitiriyor.

## Önce tanım: ajan nedir?

**Ajan**, hedefi söylediğin, yolu kendisi seçen ve elindeki araçlarla iş yapan bir programdır. Yeni bir stajyer
düşün: "şu afişi kontrol et" dersin; o önce afişe bakar, sonra müşteri dosyasını açar, sonra takvime bakar. Sırayı sen
söylemezsin. Hangi dolapları açabileceğine ise sen karar verirsin.

| | Ne yapar | Adımları kim seçer |
|---|---|---|
| Sohbet botu | Yalnızca konuşur; dosyana, takvimine dokunamaz | — |
| Üretim hattı (Konu 10) | Yazdığın adımları her seferinde aynı sırayla çalıştırır | Sen, kodda |
| Ajan (bugün) | Senin verdiğin araçları kullanarak iş yapar | Model, her istekte yeniden |

- **Araç:** senin yazdığın bir Python fonksiyonu ve onun **tarifi**. Model kodu görmez; yalnızca adını ve tarifini okur.
- **Döngü:** model bir araç ister → program aracı çalıştırır → sonuç modele döner → model ya yeni bir araç ister ya da
  cevabını yazar.
- **Ajan her zaman gerekmez.** Adımlar her seferinde aynıysa Konu 10'daki hat daha ucuzdur, daha hızlıdır ve hep aynı
  sonucu verir. Ajan, yolun isteğe göre değiştiği yerde işe yarar: yukarıdaki dört iş gibi.

Hücreleri yukarıdan aşağı sırayla çalıştır: hücreye tıkla, **Shift + Enter**. Isınma anahtarsız; Adım 1 ve sonrası
`anahtar.txt` ister (Konu 2'deki dosya). Anahtarın yoksa sonrasını hocanın ekranından izle.

## Isınma — Araç: bir fonksiyon ve tarifi

Araç sıradan bir Python fonksiyonu. Tek yeniliği, `def` satırının hemen altında üç tırnakla açılıp kapanan metin:
aracın **tarifi** (Python'daki adı *docstring*). Bugüne kadar kodlarımıza böyle açıklamalar yazmadık; burada
yazıyoruz, çünkü model aracı bu metni okuyarak seçecek. `tarih: str` ise Agno'ya "bu bilgi bir metin" der.

Aracın içi tek satır: yardımcı dosyadaki `gun_adi`'yi (Konu 10'daki gün hesabı) çağırıyor. Takvim hesabı Python'un işi.

In [ ]:
from ajan_araclari import gun_adi, tarifi_goster

def gunu_hesapla(tarih: str):
    """Bir tarihin haftanın hangi gününe denk geldiğini takvimden hesaplar. tarih: '14 Kasım 2026' gibi bir tarih."""
    return gun_adi(tarih)

gun = gunu_hesapla("14 Kasım 2026")
print(gun)

Şimdi modelin gördüğünü yazdıralım. `tarifi_goster`, Agno'nun bu fonksiyondan kurup modele göndereceği tarifi yazar:

In [ ]:
tarifi_goster(gunu_hesapla)

Modelin bildiği bu kadar: aracın adı, tarifi ve bilgisinin adı. İçindeki takvim hesabını görmez.

**Düşün, sohbete yaz:** Bu tarife bakan biri aracı aşağıdaki isteklerin hangisinde kullanır?

1. "Okuma gecesi hangi güne denk geliyor?"
2. "Afişteki mavi, kılavuzdaki maviyle aynı mı?"
3. "Matbaa cuma akşamı gelen işi pazartesi basıyor; 12 Kasım'da gönderirsem afiş ne zaman hazır olur?"

## Adım 1 — İlk ajan: tek araç

Listenin 1. işi: 14 Kasım hangi güne denk geliyor? Ajanı üç parçayla kuruyoruz: model, araç listesi ve iki ayar.

- `model_kur()` (yardımcı dosyada) Cloudflare'deki `gemma-4` modelini **Agno**'ya tanıtır. Agno, ajan döngüsünü bizim
  yerimize çalıştıran bir kütüphane. İçinde `OpenAILike` geçer ve `openai` paketini kullanır: bu yalnızca isteğin
  **biçiminin** adı. İstek Cloudflare'e gider; OpenAI hesabı da ödeme de gerekmez.
- `tools=[gunu_hesapla]`: ajanın kullanabileceği araçlar. Adı parantezsiz yazıyoruz: fonksiyonu çalıştırmıyoruz, ajana
  veriyoruz.
- `tool_call_limit=4`: bir çalıştırmada en fazla 4 araç çalışır. Model bir döngüye takılırsa ücretsiz kotan boşa gitmez.
- `telemetry=False`: Agno, bu ayar olmazsa her çalıştırmada kendi sunucusuna küçük bir kullanım kaydı gönderir; biz
  göndermiyoruz.

In [ ]:
from agno.agent import Agent
from ajan_araclari import model_kur, kararlari_goster

model = model_kur()
ajan = Agent(model=model, tools=[gunu_hesapla], tool_call_limit=4, telemetry=False)

`ajan.run` bütün döngüyü çalıştırır; birkaç saniye, bazen yarım dakika sürer. `cevap.content` ajanın son cevabı.
`kararlari_goster` ajanın kullandığı araçları sırayla yazar: aracın adı, modelin araca verdiği bilgiler ve sonucun
başı. (Agno bu listeyi `cevap.tools`'ta tutar; her aracın adı `tool_name`, bilgileri `tool_args`. Alıştırmada
bunları okuyacaksın.)

**Tahmin et, sonra çalıştır:** Ajan `gunu_hesapla`'u kullanacak mı, yoksa cevabı kendisi mi yazacak?

In [ ]:
cevap = ajan.run("14 Kasım 2026 hangi gün?")
print(cevap.content)
kararlari_goster(cevap)

Aracı kullandıysa `1. gunu_hesapla {'tarih': ...}` satırını görürsün: tarihi araca **model** yazdı, aracı **program**
çalıştırdı, sonucu modele geri verdi.

**Düşün:** Model bu soruyu araçsız, kendi bilgisiyle cevaplasaydı neden yanlış olabilirdi? Konu 10'daki afişte
"14 Kasım 2026 Cuma" yazıyordu; o gün Cumartesi.

Ekranda kırmızı `ERROR` ya da sarı `WARNING` çıkarsa hücre yine biter; ne demek olduğu `ders_notu.md`'nin "Sık
hatalar" bölümünde.

## Adım 2 — Üç araç: "bu afişi kontrol et"

Listenin 2. işi, günün en büyüğü: müşteri bu afişin kontrolünü istiyor.

<img src="veri/afis.jpg" width="300">

İki yeni araç:

- `afisi_oku`: afişteki yazıları ve renkleri okur. Konu 10'da bu adla hazır bir fonksiyon kullanmıştın; burada aynı
  işi bir araç olarak yazıyoruz, içi yardımcıdaki `gorseldeki_yazi` (gemma-4 görsele bakar).
- `kurallara_bak`: müşterinin hafızasında (kılavuz ve notlar, Konu 11) soruya anlamca en yakın beş kuralı bulur.

In [ ]:
from ajan_araclari import gorseldeki_yazi, en_yakin_kurallar

def afisi_oku(yol: str):
    """Bir afiş görselindeki bütün yazıları ve renkleri okur. yol: görselin dosya yolu, ör. veri/afis.jpg"""
    return gorseldeki_yazi(yol)

def kurallara_bak(soru: str):
    """Müşterinin marka kurallarında ve notlarında soruya en yakın beş kuralı bulur: renk, yazı, sözcük, afişte ne olmalı."""
    return en_yakin_kurallar(soru, 5)

In [ ]:
ajan = Agent(model=model, tools=[afisi_oku, gunu_hesapla, kurallara_bak], tool_call_limit=6, telemetry=False)

Üç araçlı ajan; bu iş daha çok adım isteyebilir, sınırı 6 yaptık.

**Tahmin et, sohbete yaz:** Kodun hiçbir yerinde bir sıra yazmıyor. Ajan hangi araçları, hangi sırayla kullanacak?
`gunu_hesapla`'a gerek duyacak mı?

In [ ]:
cevap = ajan.run("veri/afis.jpg afişini müşterinin kurallarına göre kontrol et")
print(cevap.content)
kararlari_goster(cevap)

Sırayı tahmininle karşılaştır. Sonra raporu afişle karşılaştır; ajanın her söylediğine güvenme:

- Bulduğu sorunlardan hangileri afişte gerçekten var? Tartışmalı bir şey yazdıysa (kılavuzun bir cümlesini harfiyen
  uygulamak gibi) kararı sen ver.
- Neyi kaçırdı? Afişte bilerek bırakılmış yedi sorun var; Konu 10 ve 11'de çoğunu buldun.
- Tarihi `gunu_hesapla`'a sordu mu? Sormadıysa yanlış günü yakalayabildi mi?
- `kurallara_bak` hangi kuralları getirdi (`sonuç:` satırı)? Hepsi afişle ilgili mi? Konu 11'den hatırla: arama
  her zaman en yakın olanları getirir, ilgili olmasalar bile.

## Adım 3 — "Unutma": hafızaya not yazdırmak

Listenin 3. işi: müşteri hardal sarısını da artık istemiyor. Asistan bunu bir sonraki kontrolde bilmeli. Ama ajan
iki `ajan.run` arasında konuşmayı hatırlamaz; her çalıştırma sıfırdan başlar. Hatırlamasını istediğin şeyi bir
dosyaya yazdırırsın.

Asistanın hafızası `ciktilar/hafiza.txt`: kılavuzun ve notların (Konu 11) bir kopyası, her satırı bir kural ya da not.
İlk kullanıldığında (Adım 2'de, `kurallara_bak` ile) açıldı. `hatirla` aracı bu dosyanın sonuna bir satır ekler;
`veri/` içindeki dosyalar değişmez.

In [ ]:
from ajan_araclari import hafizaya_ekle

def hatirla(not_metni: str):
    """Müşterinin yeni bir isteğini ya da kuralını hafızaya kalıcı olarak yazar. Kullanıcı 'unutma' ya da 'not al' derse kullan."""
    return hafizaya_ekle(not_metni)

araclar = [afisi_oku, gunu_hesapla, kurallara_bak, hatirla]
ajan = Agent(model=model, tools=araclar, tool_call_limit=6, telemetry=False)

**Tahmin et:** Ajan hangi aracı seçecek? Notu hangi cümleyle yazacak?

In [ ]:
cevap = ajan.run("Müşteri artık afişlerde hardal sarısı da istemiyor, unutma.")
print(cevap.content)
kararlari_goster(cevap)

Hafıza büyüdü mü? `hafizayi_oku` hafızayı satır satır bir liste olarak verir; `notlar[-1]` listenin son elemanı.

In [ ]:
from ajan_araclari import hafizayi_oku

notlar = hafizayi_oku()
sayi = len(notlar)
print(sayi, "satır")
print(notlar[-1])

Şimdi Adım 2'deki isteği aynen tekrarla. Kodu değiştirmedik; değişen yalnızca hafıza. Bu kez hardal sarısı yazıdan
söz edecek mi? Kılavuz hardalı "küçük vurgu" olarak serbest bırakıyor, yeni not yasaklıyor: ajan hangisine uydu?

In [ ]:
cevap = ajan.run("veri/afis.jpg afişini müşterinin kurallarına göre kontrol et")
print(cevap.content)
kararlari_goster(cevap)

Araçlara da bak: Adım 2'dekiyle aynı araçları, aynı sırayla mı seçti? Ajan her çalıştırmada yolu yeniden seçer; aynı
istekte bir kez `gunu_hesapla`'u atlayıp bir kez çağırabilir. Konu 10'daki hat ise her afişte aynı adımları çalıştırır,
tarihi hiç atlamaz. Her afişte aynı kontrol gerekiyorsa hat; iş her gün değişiyorsa ajan.

Bu, müşteriye gidecek kontrol notunun taslağı. Dosyaya kaydet; okuyup düzeltmek ve göndermek senin işin.

In [ ]:
with open("ciktilar/afis_kontrolu.txt", "w", encoding="utf-8") as dosya:
    dosya.write(cevap.content)

## Adım 4 — Kendi aracın: ad ve tarif bir brif

Listenin son işi: gönderi metni müşterinin kurallarına uyuyor mu? Müşteri metinlerde bazı sözcükleri istemiyor
(kılavuzda "indirim", "kampanya", "fırsat", "hemen"; notlarda "kafe"). Bir sözcüğün metinde geçip geçmediğine bakmak
sıradan Python'un işi: model bir sözcüğü gözden kaçırabilir, Python kaçırmaz. (Adım 2'deki raporda afişteki
"kafemizde" yakalandı mı?) Bu aracı sen yazacaksın.

**Kendi elinle:** aşağıdaki hücrede `bulunan = []` satırının altındaki boş satıra, aynı girintiyle üç satır yaz:

1. `yasaklar` listesindeki her `sozcuk` için,
2. sözcük `metin.lower()` içinde geçiyorsa (`lower()` metni küçük harfe çevirir: "Hemen" → "hemen"),
3. onu `bulunan`'a ekle: `bulunan.append(sozcuk)`.

Hücreyi çalıştır: `['kafe', 'indirim', 'hemen']` görmelisin; `[]` görüyorsan döngü henüz yok. İstersen listeye kendi
müşterinin bir sözcüğünü ekle. Aracın adını ve tarifini bilerek tembel koyduk: `metne_bak`, "Metne bakar."

In [ ]:
def metne_bak(metin: str):
    """Metne bakar."""
    yasaklar = ["kafe", "indirim", "kampanya", "fırsat", "hemen"]
    bulunan = []

    return bulunan

deneme = metne_bak("Hemen gel, kafemizde sınav haftası indirimi!")
print(deneme)
tarifi_goster(metne_bak)

**Tahmin et, sohbete yaz:** Ajanın elinde artık beş araç var. Aşağıdaki istekte senin aracını mı seçecek,
`kurallara_bak`'ı mı, ikisini birden mi, hiçbirini mi?

In [ ]:
araclar = [afisi_oku, gunu_hesapla, kurallara_bak, hatirla, metne_bak]
ajan = Agent(model=model, tools=araclar, tool_call_limit=6, telemetry=False)

cevap = ajan.run("Bu gönderi metni müşterinin kurallarına uyuyor mu: 'Hemen gel, kafemizde sınav haftası indirimi!'")
print(cevap.content)
kararlari_goster(cevap)

Hangisini seçti, sohbete yaz. "Metne bakar" bir müşterinin "bir şey tasarla" demesi gibi: muğlak brif yanlış iş
getirir. Model aracı adından ve tarifinden tanır; ikisi de bir şey söylemiyorsa aracı seçmeyebilir ya da yanlış yerde
seçebilir.

**Brifi düzelt:** yukarıdaki `metne_bak` hücresinde "Metne bakar." yerine kendi tarifini yaz. Bir brif gibi üç şey
söylesin: araç ne yapar, ne zaman kullanılmalı, `metin` bilgisine ne yazılmalı. O hücreyi yeniden çalıştır (`Tarifi`
satırında yeni cümlen görünmeli), sonra aşağıdaki hücreyi. Aradaki ajan hücresine dokunma: ilk sonuç karşılaştırma
için orada dursun.

In [ ]:
araclar = [afisi_oku, gunu_hesapla, kurallara_bak, hatirla, metne_bak]
ajan = Agent(model=model, tools=araclar, tool_call_limit=6, telemetry=False)

cevap = ajan.run("Bu gönderi metni müşterinin kurallarına uyuyor mu: 'Hemen gel, kafemizde sınav haftası indirimi!'")
print(cevap.content)
kararlari_goster(cevap)

Bu kez senin aracını seçti mi? Seçmediyse tarifini değiştir, `def` hücresini ve bu son ajan hücresini yeniden
çalıştır. Yalnızca `def`i çalıştırmak yetmez: araç listesi ve ajan, ajan hücresinde yeniden kuruluyor. Muğlak tarifle
de seçtiyse bu bir tahmindi, başka bir istekte tutmayabilir. Ödevde aracına işini söyleyen bir ad da ver: `metne_bak`
yerine `yasak_sozcuk_bul` gibi.

## Bonus — Yarışma ve staj takipçisi

Portfolyon için yarışmaları takip etmek sıkıcı ve kolay unutulan bir iş. Bu ajanın tek aracı ücretsiz bir web araması
(`ddgs` paketi; anahtar istemez). `add_datetime_to_context=True` ajana bugünün tarihini söyler: model bugünün
tarihini kendiliğinden bilmez.

In [ ]:
from ajan_araclari import internette_ara

def webde_ara(sorgu: str):
    """İnternette tek bir arama yapar; ilk beş sonucun başlığını, adresini ve kısa özetini verir. sorgu: aranacak sözcükler, Türkçe. Her çağrıda yalnız bir sorgu gönder."""
    return internette_ara(sorgu)

ajan = Agent(model=model, tools=[webde_ara], tool_call_limit=5, telemetry=False, add_datetime_to_context=True)

In [ ]:
istek = "Tek bir arama yap. Son başvurusu önümüzdeki iki ay içinde olan afiş ya da grafik tasarım yarışmalarını bul. Her birinin adını, son tarihini ve bağlantısını yaz; son tarih arama sonucunda yoksa 'bilinmiyor' yaz."
cevap = ajan.run(istek)
print(cevap.content)
kararlari_goster(cevap)

**Listedeki her bağlantıya tıkla.** Model arama sonuçlarını özetlerken yarışma adı ya da tarih uydurabilir; listede
"tahmini" diye bir tarih varsa o tarih bir sitede değil, modelin tahmininde. Her yarışmanın adını ve son tarihini kendi
sitesinden doğrula; doğrulamadığını listene alma.

Kendi isteğinle dene: "İzmir'de grafik tasarım stajı ilanları", "öğrencilere açık logo yarışmaları"...

---

**Bugün:** Sayfa Kahve'nin dört işini bir asistanla bitirdin. Hedefi sen söyledin, yolu ajan seçti; araçların içini
sen yazdın, kararlarını `kararlari_goster` ile okudun. Ödev: `odevler/odev12.md`.